# Phase 2 — Multi-Seed, Multi-Encoder Training Matrix

Chapter 4 currently reports a **single run** of a single encoder. A single run gives no
handle on seed variance, so it cannot support a claim that one encoder beats another. This
notebook replaces the point estimates with mean +/- sd over three seeds per configuration
and settles the encoder question on our own data.

**The encoder question.** A panelist's published work reports XLNet outperforming BERT on
privacy-policy classification; the thesis claims legal-domain pretraining dominates. Both
claims are about *other* corpora. Running all four encoders through an identical protocol
on identical splits is the only way to know which holds here.

## Run matrix

| Axis | Values |
| --- | --- |
| Encoders (dual-head) | `nlpaueb/legal-bert-base-uncased`, `bert-base-uncased`, `xlnet-base-cased`, `roberta-base` |
| Seeds | 42, 1337, 2024 |
| Head ablation (**all four encoders**) | topic-only, risk-only (dual-head reuses the runs above) |

**12 dual-head runs + 24 ablation runs = 36 (4 encoders x 3 head configs x 3 seeds).**

The ablation was originally run for legal-bert only, on the argument that "does joint
training help?" is a question about the architecture, not the backbone. That was an
assumption, not a measurement, and compute is no longer the binding constraint, so it is
now tested directly: the ablation is run for all four encoders and the dual-vs-single-head
delta is compared across them. XLNet is the encoder most likely to break the pattern — its
seed-to-seed sd is 5-10x every BERT-family encoder's and it pools its summary token from
the last position rather than the first — so an "architecture, not backbone" claim has to
survive XLNet to be worth making. The cross-encoder consistency check at the end of the
notebook is the deliverable; the extra 18 runs only exist to feed it.

## What is held fixed

Everything except encoder / seed / head mode, and it is held fixed by construction — the
runner imports `scripts/lawgic_train_matrix.py`, which reads the same persisted seed-42
split file, the same taxonomy, the same masked-BCE + masked-CE losses (copied line for
line from the original `DualHeadTrainer`), lr 3e-5, batch 8, up to 20 epochs, early
stopping patience 3, weight decay 0.01, warmup 0.06, FP16 on CUDA, max_length 256, and the
same pre-training degenerate-model assertion (a zero-logit model must score topic macro-F1
below 0.95).

## How the pooled representation is chosen per architecture

The two linear heads read one vector per clause. Which token that vector comes from is
**not** the same across these four encoders, and getting it wrong silently cripples a
model rather than erroring:

- **BERT, Legal-BERT, RoBERTa** — the sequence summary is the **first** token
  (`[CLS]` / `<s>`), placed there during pretraining.
- **XLNet** — XLNet is trained with the summary token **appended at the end**. Reading
  position 0 would hand the head an ordinary content token. So XLNet uses the **last**
  token.

`pooled_representation()` in `scripts/lawgic_train_matrix.py` is the single place this
lives. It selects by attention mask rather than by fixed index (`attention_mask.argmax(1)`
for first, `L - 1 - flip(mask).argmax(1)` for last), because XLNet's tokenizer pads on the
**left** while the BERT-family tokenizers pad on the right — a hardcoded `[:, 0]` or
`[:, -1]` would read padding for one of them.

Two further per-architecture quirks are handled in the same adapter, not scattered around:

- **RoBERTa has no `token_type_ids`.** The collator keeps only the keys in
  `tokenizer.model_input_names`, so each tokenizer declares its own contract and no
  `if roberta:` branch is needed anywhere.
- **XLNet's tokenizer needs `sentencepiece`.** Already present in
  `notebooks/requirements.txt` (`sentencepiece==0.2.1`); listed as a manual check below.

**Deviation to record in the manuscript.** The original v3 checkpoint fed the heads BERT's
`pooler_output` (a dense+tanh layer on top of `[CLS]`). The matrix uses the raw first
token instead, for all encoders. Reason: `roberta-base` ships with a *randomly initialised*
pooler, so keeping `pooler_output` would have handicapped RoBERTa for reasons unrelated to
the encoder itself. Consistency across the four arms matters more than bit-matching the
old run, so the legal-bert/seed-42 cell of this matrix is **not** expected to reproduce the
v3 checkpoint exactly — treat the matrix as internally comparable and the Phase 1 numbers
as the checkpoint's own.

In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    sentinel = Path("generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv")
    for candidate in (start, *start.parents):
        if (candidate / sentinel).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

split_path = core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS
print(f"Split artifact: {split_path}")
print("Rows:", {k: len(v) for k, v in frames.items()})

MATRIX = tm.build_matrix()
print(f"\nConfigured runs: {len(MATRIX)}")
display(pd.DataFrame([{
    "run_id": c.run_id, "encoder": c.encoder_name, "seed": c.seed,
    "heads": c.heads, "selection_metric": c.best_metric_key,
} for c in MATRIX]))

Split artifact: C:\Users\Enrique\Coding Projects\Thesis\lawgic\generated_files\lawgic_taxonomy\splits\split_seed42.csv
Rows: {'train': 21183, 'validation': 2648, 'test': 2648}

Configured runs: 18


,run_id,encoder,seed,heads,selection_metric
0,legal-bert-base-uncased__seed42__dual,nlpaueb/legal-bert-base-uncased,42,dual,topic_macro_f1
1,legal-bert-base-uncased__seed1337__dual,nlpaueb/legal-bert-base-uncased,1337,dual,topic_macro_f1
2,legal-bert-base-uncased__seed2024__dual,nlpaueb/legal-bert-base-uncased,2024,dual,topic_macro_f1
3,bert-base-uncased__seed42__dual,bert-base-uncased,42,dual,topic_macro_f1
4,bert-base-uncased__seed1337__dual,bert-base-uncased,1337,dual,topic_macro_f1
5,bert-base-uncased__seed2024__dual,bert-base-uncased,2024,dual,topic_macro_f1
6,xlnet-base-cased__seed42__dual,xlnet-base-cased,42,dual,topic_macro_f1
7,xlnet-base-cased__seed1337__dual,xlnet-base-cased,1337,dual,topic_macro_f1
8,xlnet-base-cased__seed2024__dual,xlnet-base-cased,2024,dual,topic_macro_f1
9,roberta-base__seed42__dual,roberta-base,42,dual,topic_macro_f1


### Extending the matrix

`build_matrix()` returns the original 18 runs (4 encoders x 3 seeds dual-head, plus
legal-bert topic-only/risk-only x 3 seeds). `MATRIX` is a list of `RunConfig` dataclasses,
so extra arms are appended here rather than by rewriting `build_matrix()` — the function
stays the record of what Phase 2 originally ran.

The cell below appends the **18 missing ablation runs**: topic-only and risk-only, three
seeds each, for BERT, XLNet and RoBERTa. `RunConfig.best_metric_key` gives risk-only runs
`risk_macro_f1` and everything else `topic_macro_f1` automatically, so the selection
asymmetry that the legal-bert ablation already uses is replicated for the new encoders by
construction, not by hand. Nothing else about the protocol is touched.

The commented line keeps the earlier five-seed option available. Do **not** add seeds to
only some arms and then compare sds across arms — the sd of 5 draws is not comparable to
the sd of 3.

In [2]:
MATRIX += [
    tm.RunConfig(encoder_name=encoder, seed=seed, heads=heads)
    for encoder in tm.ENCODERS[1:]
    for heads in ("topic", "risk")
    for seed in tm.SEEDS
]
assert len(MATRIX) == 36 and len({c.run_id for c in MATRIX}) == 36, len(MATRIX)
print(f"Runs after extension: {len(MATRIX)}")
display(pd.DataFrame([{
    "run_id": c.run_id, "encoder": c.encoder_name, "seed": c.seed,
    "heads": c.heads, "selection_metric": c.best_metric_key,
} for c in MATRIX if c.heads != "dual"]))

# MATRIX += [tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=s, heads="dual") for s in (7, 2718)]


Runs after extension: 36


,run_id,encoder,seed,heads,selection_metric
0,legal-bert-base-uncased__seed42__topic,nlpaueb/legal-bert-base-uncased,42,topic,topic_macro_f1
1,legal-bert-base-uncased__seed1337__topic,nlpaueb/legal-bert-base-uncased,1337,topic,topic_macro_f1
2,legal-bert-base-uncased__seed2024__topic,nlpaueb/legal-bert-base-uncased,2024,topic,topic_macro_f1
3,legal-bert-base-uncased__seed42__risk,nlpaueb/legal-bert-base-uncased,42,risk,risk_macro_f1
4,legal-bert-base-uncased__seed1337__risk,nlpaueb/legal-bert-base-uncased,1337,risk,risk_macro_f1
5,legal-bert-base-uncased__seed2024__risk,nlpaueb/legal-bert-base-uncased,2024,risk,risk_macro_f1
6,bert-base-uncased__seed42__topic,bert-base-uncased,42,topic,topic_macro_f1
7,bert-base-uncased__seed1337__topic,bert-base-uncased,1337,topic,topic_macro_f1
8,bert-base-uncased__seed2024__topic,bert-base-uncased,2024,topic,topic_macro_f1
9,bert-base-uncased__seed42__risk,bert-base-uncased,42,risk,risk_macro_f1


## MANUAL STEP — before running the matrix

1. **Model downloads.** The first run of each encoder pulls weights from the HuggingFace
   hub (~440 MB each for `bert-base-uncased`, `xlnet-base-cased`, `roberta-base`;
   legal-bert is already local). Requires network access on the training machine. The
   cell below pre-fetches all three in-notebook via `AutoModel`/`AutoTokenizer`.
2. **`sentencepiece`** must be importable for the XLNet tokenizer. It is already in
   `notebooks/requirements.txt`; the check cell below verifies it rather than installing it.
3. **GPU.** These are 36 full fine-tunes. On CPU this is days, not hours — run on the CUDA
   machine that produced the v3 checkpoint. FP16 switches on automatically on CUDA and off
   elsewhere, matching the original protocol.
4. **Disk.** Each run keeps one checkpoint (`save_total_limit=1`), ~440 MB, plus a small
   `test_logits.npz`. Budget ~20 GB for the full matrix under
   `generated_files/lawgic_taxonomy/runs/`.

Nothing here writes to `saved_models/`; the deployed v3 checkpoint is never touched.

In [5]:
from transformers import AutoModel, AutoTokenizer

MODELS_TO_PREFETCH = ["bert-base-uncased", "xlnet-base-cased", "roberta-base"]

for model_name in MODELS_TO_PREFETCH:
    print(f"Downloading {model_name} ...")
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f"  done: {model_name}")

print("\nAll three encoders cached locally.")

  done: bert-base-uncased
  done: xlnet-base-cased


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  done: roberta-base

All three encoders cached locally.


In [9]:
%conda install conda-forge::sentencepiece

3 channel Terms of Service accepted
Channels:
 - defaults
 - conda-forge
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\Enrique\anaconda3\envs\thesis-env

  added / updated specs:
    - conda-forge::sentencepiece


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libabseil-20260526.0       | cxx17_h4cdcee1_0         1.9 MB
    libprotobuf-7.35.1         |       hb5abd84_0         6.9 MB
    libsentencepiece-0.2.1     |       h1e80020_4         1.4 MB  conda-forge
    sentencepiece-0.2.1        |       hb9477dd_4          20 KB  conda-forge
    sentencepiece-python-0.2.1 |  py314h2f88111_4         3.2 MB  conda-forge
    sentencepiece-spm-0.2.1    |       h1e80020_4         173 KB  conda-forge
    ------------------------------------------------------------
                                           Total:        13.5 MB

The following NEW 



==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [3]:
import importlib.util

print("sentencepiece:", "OK" if importlib.util.find_spec("sentencepiece") else "MISSING — XLNet will fail")
print("scipy:", "OK" if importlib.util.find_spec("scipy") else "MISSING — McNemar will fail")

device_label, device = tm.detect_device()
print(f"device: {device_label} (fp16={device_label == 'cuda'})")
if device_label != "cuda":
    print("WARNING: not on CUDA. The matrix will take days. Stop and move to the GPU machine.")

sentencepiece: OK
scipy: OK
device: cuda (fp16=True)


## Expected wall time

The v3 run's `trainer_state.json` records **17 epochs** before early stopping (best at
epoch 14, patience 3) at batch size 8 over 21,183 training rows = 2,648 optimizer steps
per epoch, and an eval throughput of ~455 clauses/s on the original CUDA device. It does
**not** record `train_runtime` — the notebook that produced it never logged the summary —
so the per-run wall time must be **measured on the first run**, not assumed.

The cell below prints the derived lower bound from what *is* recorded, then the runner
stores the real `wall_seconds` for every run. After the first run completes, multiply.

In [ ]:
state_path = core.CHECKPOINT_DIR / "checkpoints/checkpoint-45016/trainer_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    evals = [h for h in state["log_history"] if "eval_runtime" in h]
    eval_throughput = float(np.mean([h["eval_samples_per_second"] for h in evals]))
    epochs = float(state["epoch"])
    # Training is roughly 3-4x the cost of inference per sample (forward + backward + optimizer).
    optimistic_seconds = epochs * (len(frames["train"]) / (eval_throughput / 3.5))
    print(f"v3 run: {epochs:.0f} epochs, eval throughput {eval_throughput:.0f} clauses/s")
    print(f"Derived LOWER BOUND per run: ~{optimistic_seconds / 60:.0f} min "
          f"-> ~{len(MATRIX) * optimistic_seconds / 3600:.1f} h for {len(MATRIX)} runs")
    print("This is an extrapolation, not a measurement. Trust wall_seconds from run 1 instead.")
else:
    print("No v3 trainer_state.json found; wall time must be measured on the first run.")

## Runner

Each config trains, evaluates on the frozen test split, and writes to
`generated_files/lawgic_taxonomy/runs/<run_id>/`:

- `metrics.json` — config + headline test metrics + `wall_seconds` + `epochs_run`
- `test_logits.npz` — test logits, labels and masks (so aggregation, bootstrap and paired
  tests never need to re-run inference)
- `per_topic.csv` — per-topic precision / recall / F1 / support

Completed runs are skipped, so the cell is **resumable**: interrupt it, restart the kernel,
re-run. Set `FORCE_RERUN = True` to redo everything.

### Best-model export helper

Defined before the runner so each encoder's best checkpoint is written to `saved_models/` as soon as its runs finish, rather than only after all 36.

In [5]:
# Best-model export, defined BEFORE the runner so the matrix can save each
# encoder's best checkpoint to saved_models/ as soon as that encoder's runs
# finish, instead of only after all 36 runs complete. Interrupting the matrix
# therefore never loses an already-trained model.
#
# Layout matches lawgic_classifier_legal-bert_v3 (model_state_dict.pt +
# encoder/tokenizer + head weights + taxonomy + metadata). Reads metrics.json
# directly from disk, so it is resumable across kernel restarts. Writes to a NEW
# directory per encoder (suffix "_phase2") - never touches
# lawgic_classifier_legal-bert_v3.
#
# Called after every dual-head run, so the export is redone when a later seed
# beats the currently exported one; if the exported directory already holds the
# best run it is left untouched.

import shutil
from datetime import datetime, timezone

import torch
from safetensors.torch import load_file as load_safetensors
from transformers import AutoTokenizer

SAVE_TARGETS = {
    "nlpaueb/legal-bert-base-uncased": "legal-bert",
    "bert-base-uncased": "bert",
    "xlnet-base-cased": "xlnet",
    "roberta-base": "roberta",
}
SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models"


def completed_dual_runs(encoder_name: str) -> list[dict]:
    records = []
    for metrics_path in sorted(tm.RUNS_DIR.glob("*/metrics.json")):
        record = json.loads(metrics_path.read_text())
        if record["encoder_name"] == encoder_name and record["heads"] == "dual":
            records.append(record)
    return records


def best_checkpoint_dir(run_id: str) -> Path:
    checkpoints = sorted(
        (tm.RUNS_DIR / run_id / "checkpoints").glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint saved for {run_id}")
    # save_total_limit=1 + load_best_model_at_end=True: the one surviving
    # checkpoint is the best validation checkpoint, not just the last epoch.
    return checkpoints[-1]


def save_best_model(encoder_name: str, short_name: str | None = None) -> None:
    short_name = short_name or SAVE_TARGETS[encoder_name]
    candidates = completed_dual_runs(encoder_name)
    if not candidates:
        print(f"skip {short_name}: no completed dual-head runs yet")
        return

    best = max(candidates, key=lambda r: r["best_val_metric"])
    run_id = best["run_id"]

    output_dir = SAVED_MODELS_DIR / f"lawgic_classifier_{short_name}_phase2"
    if output_dir.exists():
        existing = output_dir / "training_metadata.json"
        exported_run = (
            json.loads(existing.read_text())["source_run_id"] if existing.exists() else None
        )
        if exported_run == run_id:
            print(f"skip {short_name}: {run_id} already exported")
            return
        print(f"[{short_name}] {exported_run} superseded by {run_id}, re-exporting")
        shutil.rmtree(output_dir)

    checkpoint_dir = best_checkpoint_dir(run_id)

    model = tm.LawgicDualHeadModel(encoder_name)
    weights_file = checkpoint_dir / "model.safetensors"
    state_dict = (
        load_safetensors(str(weights_file))
        if weights_file.exists()
        else torch.load(checkpoint_dir / "pytorch_model.bin", map_location="cpu", weights_only=True)
    )
    model.load_state_dict(state_dict)

    tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_dir))

    output_dir.mkdir(parents=True)

    # Full state dict + encoder/tokenizer + heads separately, mirroring v3's layout.
    torch.save(model.state_dict(), output_dir / "model_state_dict.pt")
    model.encoder.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    torch.save(model.topic_head.state_dict(), output_dir / "topic_head_weights.pt")
    torch.save(model.harm_head.state_dict(), output_dir / "harm_head_weights.pt")

    topic_ids, name_by_topic, _ = core.load_taxonomy()
    compact_taxonomy = [
        {"classifier_id": i, "topic_id": tid, "name": name_by_topic[tid]}
        for i, tid in enumerate(topic_ids)
    ]
    (output_dir / "lawgic_topics_44.json").write_text(json.dumps(compact_taxonomy, indent=2))
    shutil.copy2(core.TAXONOMY_PATH, output_dir / "lawgic_topics_original_45.json")

    (output_dir / "test_metrics.json").write_text(json.dumps(best, indent=2, default=str))

    metadata = {
        "model_name": encoder_name,
        "architecture": "dual_head",
        "num_topics": core.NUM_LAWGIC_TOPICS,
        "num_harm_classes": core.NUM_HARM_CLASSES,
        "max_length": core.MAX_LENGTH,
        "decision_threshold": core.DECISION_THRESHOLD,
        "seed": best["seed"],
        "source_run_id": run_id,
        "best_val_metric": best["best_val_metric"],
        "seeds_considered": sorted(r["seed"] for r in candidates),
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "note": (
            "Best-of-3-seeds model from the Phase 2 multi-encoder matrix "
            "(notebooks/evaluation/02_multiseed_encoder_runs.ipynb); does not "
            "replace lawgic_classifier_legal-bert_v3."
        ),
    }
    (output_dir / "training_metadata.json").write_text(json.dumps(metadata, indent=2))

    print(f"[{short_name}] saved best seed {best['seed']} (run {run_id}) -> {output_dir}")



In [8]:
FORCE_RERUN = False

records = []
for index, config in enumerate(MATRIX, start=1):
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists() and not FORCE_RERUN:
        print(f"[{index}/{len(MATRIX)}] skip {config.run_id} (already complete)")
        records.append(json.loads(target.read_text()))
    else:
        print(f"[{index}/{len(MATRIX)}] running {config.run_id} ...")
        records.append(tm.run_config(config))
    # Export this encoder's best dual-head run as soon as it is known, so an
    # interrupted matrix keeps every model it has already trained. Runs on the
    # skip branch too: the dual arms may already be on disk from an earlier
    # session, and the export is idempotent (it compares source_run_id).
    if config.heads == "dual":
        save_best_model(config.encoder_name)

print(f"Completed {len(records)} runs.")

[1/36] skip legal-bert-base-uncased__seed42__dual (already complete)
skip legal-bert: legal-bert-base-uncased__seed2024__dual already exported
[2/36] skip legal-bert-base-uncased__seed1337__dual (already complete)
skip legal-bert: legal-bert-base-uncased__seed2024__dual already exported
[3/36] skip legal-bert-base-uncased__seed2024__dual (already complete)
skip legal-bert: legal-bert-base-uncased__seed2024__dual already exported
[4/36] skip bert-base-uncased__seed42__dual (already complete)
skip bert: bert-base-uncased__seed2024__dual already exported
[5/36] skip bert-base-uncased__seed1337__dual (already complete)
skip bert: bert-base-uncased__seed2024__dual already exported
[6/36] skip bert-base-uncased__seed2024__dual (already complete)
skip bert: bert-base-uncased__seed2024__dual already exported
[7/36] skip xlnet-base-cased__seed42__dual (already complete)
skip xlnet: xlnet-base-cased__seed2024__dual already exported
[8/36] skip xlnet-base-cased__seed1337__dual (already complete)


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.078600,0.142293,0.532536,0.731158,0.706319,96620.000000,0.490559,0.340338,0.396045,2648.000000
2,0.050500,0.109446,0.654715,0.800797,0.792602,96620.000000,0.458837,0.406741,0.431890,2648.000000
3,0.033400,0.106686,0.712460,0.817105,0.812094,96620.000000,0.452417,0.385874,0.411136,2648.000000
4,0.030200,0.104776,0.743192,0.822634,0.822059,96620.000000,0.384819,0.329305,0.356712,2648.000000
5,0.024600,0.103574,0.740277,0.819638,0.818267,96620.000000,0.394260,0.338967,0.366873,2648.000000
6,0.010600,0.102919,0.742810,0.827888,0.826787,96620.000000,0.424849,0.346059,0.382547,2648.000000
7,0.010800,0.105509,0.746794,0.830426,0.827972,96620.000000,0.386707,0.314662,0.356834,2648.000000
8,0.008400,0.128403,0.738067,0.827886,0.826989,96620.000000,0.375755,0.331558,0.361876,2648.000000
9,0.007500,0.131851,0.735316,0.822447,0.819924,96620.000000,0.363293,0.304075,0.341443,2648.000000
10,0.003300,0.133853,0.737506,0.823878,0.822576,96620.000000,0.390106,0.321319,0.354464,2648.000000


[bert-base-uncased__seed42__topic] 32.4 min | topic_macro_f1=0.7471 topic_micro_f1=0.8340 risk_accuracy=nan risk_macro_f1=nan
[20/36] running bert-base-uncased__seed1337__topic ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


...

Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.502000,0.529084,0.077256,0.091386,0.117533,96620.000000,0.787009,0.776604,0.786891,2648.000000
2,0.488700,0.502977,0.074308,0.089403,0.116013,96620.000000,0.824773,0.821231,0.825015,2648.000000
3,0.410900,0.516103,0.087209,0.092230,0.134306,96620.000000,0.832704,0.827136,0.832998,2648.000000
4,0.334900,0.571187,0.083471,0.093572,0.130510,96620.000000,0.836480,0.831075,0.835494,2648.000000
5,0.314300,0.769231,0.106053,0.096766,0.148832,96620.000000,0.825529,0.817396,0.825364,2648.000000
6,0.347700,0.807377,0.101826,0.096353,0.145915,96620.000000,0.827795,0.822530,0.827155,2648.000000
7,0.344300,0.928525,0.095826,0.102043,0.146048,96620.000000,0.827039,0.821057,0.826968,2648.000000


[roberta-base__seed1337__risk] 24.3 min | topic_macro_f1=nan topic_micro_f1=nan risk_accuracy=0.8240 risk_macro_f1=0.8161
[36/36] running roberta-base__seed2024__risk ...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.584400,0.676103,0.083064,0.099350,0.128269,96620.000000,0.779456,0.766645,0.775911,2648.000000
2,0.493900,0.475468,0.084333,0.099757,0.123931,96620.000000,0.818353,0.812516,0.818595,2648.000000
3,0.390700,0.576033,0.082695,0.101831,0.120939,96620.000000,0.818353,0.809160,0.816006,2648.000000
4,0.528100,0.575839,0.087970,0.104236,0.142732,96620.000000,0.827795,0.822030,0.827641,2648.000000
5,0.440700,0.655995,0.083651,0.100396,0.130586,96620.000000,0.838746,0.836012,0.839259,2648.000000
6,0.334200,0.804718,0.087600,0.101128,0.140843,96620.000000,0.829305,0.826910,0.830109,2648.000000
7,0.315200,0.824453,0.091180,0.102913,0.146812,96620.000000,0.842145,0.839614,0.842833,2648.000000
8,0.182100,1.013039,0.089599,0.100773,0.145278,96620.000000,0.836858,0.831228,0.836591,2648.000000
9,0.131300,1.066385,0.090115,0.100963,0.143799,96620.000000,0.829683,0.824228,0.829929,2648.000000
10,0.252700,1.052193,0.095297,0.108570,0.158305,96620.000000,0.830816,0.823684,0.829448,2648.000000


[roberta-base__seed2024__risk] 34.5 min | topic_macro_f1=nan topic_micro_f1=nan risk_accuracy=0.8089 risk_macro_f1=0.8020
Completed 36 runs.


Re-export the best models (no-op if the runner already did)

In [9]:
# Safety net: after the full matrix, confirm every encoder's best dual-head run
# is the one exported. A no-op when the in-loop export already did it, and the
# way models get written if the runner cell was skipped entirely.
for encoder_name, short_name in SAVE_TARGETS.items():
    save_best_model(encoder_name, short_name)


skip legal-bert: legal-bert-base-uncased__seed2024__dual already exported
skip bert: bert-base-uncased__seed2024__dual already exported
skip xlnet: xlnet-base-cased__seed2024__dual already exported
skip roberta: roberta-base__seed2024__dual already exported


## Aggregation

Everything below reads the persisted run artifacts, so it can be re-run without a GPU.

In [10]:
run_files = sorted(tm.RUNS_DIR.glob("*/metrics.json"))
runs = pd.DataFrame([json.loads(p.read_text()) for p in run_files])
runs = runs[runs["holdout_source"].isna()] if "holdout_source" in runs else runs
print(f"Loaded {len(runs)} Phase 2 runs from {tm.RUNS_DIR}")
display(runs[["run_id", "encoder_name", "seed", "heads", "epochs_run", "wall_seconds", *core.HEADLINE_METRICS]])

runs.to_csv(core.EVAL_OUT_DIR / "phase2_runs.csv", index=False)
print(f"\nMeasured wall time: {runs['wall_seconds'].mean() / 60:.1f} min/run "
      f"(total {runs['wall_seconds'].sum() / 3600:.1f} h)")

Loaded 36 Phase 2 runs from C:\Users\Enrique\Coding Projects\Thesis\lawgic\generated_files\lawgic_taxonomy\runs


,run_id,encoder_name,seed,heads,epochs_run,wall_seconds,topic_macro_f1,topic_micro_f1,risk_accuracy,risk_macro_f1
0,bert-base-uncased__seed1337__dual,bert-base-uncased,1337,dual,15.0,2950.104609,0.768129,0.833610,0.826284,0.819565
1,bert-base-uncased__seed1337__risk,bert-base-uncased,1337,risk,7.0,1384.060199,NaN,NaN,0.826284,0.819923
2,bert-base-uncased__seed1337__topic,bert-base-uncased,1337,topic,8.0,1572.422819,0.737961,0.821785,NaN,NaN
3,bert-base-uncased__seed2024__dual,bert-base-uncased,2024,dual,20.0,3875.743355,0.780256,0.836836,0.836480,0.831212
4,bert-base-uncased__seed2024__risk,bert-base-uncased,2024,risk,9.0,1782.525506,NaN,NaN,0.812689,0.808053
5,bert-base-uncased__seed2024__topic,bert-base-uncased,2024,topic,11.0,2132.144447,0.754520,0.826038,NaN,NaN
6,bert-base-uncased__seed42__dual,bert-base-uncased,42,dual,20.0,3820.460032,0.775652,0.832867,0.826284,0.820721
7,bert-base-uncased__seed42__risk,bert-base-uncased,42,risk,9.0,1790.631732,NaN,NaN,0.812311,0.808309
8,bert-base-uncased__seed42__topic,bert-base-uncased,42,topic,10.0,1945.131732,0.747051,0.834038,NaN,NaN
9,legal-bert-base-uncased__seed1337__dual,nlpaueb/legal-bert-base-uncased,1337,dual,14.0,2716.261310,0.769291,0.832933,0.837991,0.832859



Measured wall time: 50.0 min/run (total 30.0 h)


In [11]:
grouped = runs.groupby(["encoder_name", "heads"])
aggregate = grouped[list(core.HEADLINE_METRICS)].agg(["mean", "std", "count"])
aggregate.columns = ["_".join(c) for c in aggregate.columns]
aggregate = aggregate.reset_index()
display(aggregate)
aggregate.to_csv(core.EVAL_OUT_DIR / "phase2_aggregate.csv", index=False)

,encoder_name,heads,topic_macro_f1_mean,topic_macro_f1_std,topic_macro_f1_count,topic_micro_f1_mean,topic_micro_f1_std,topic_micro_f1_count,risk_accuracy_mean,risk_accuracy_std,risk_accuracy_count,risk_macro_f1_mean,risk_macro_f1_std,risk_macro_f1_count
0,bert-base-uncased,dual,0.774679,0.006122,3,0.834438,0.002110,3,0.829683,0.005887,3,0.823833,0.006416,3
1,bert-base-uncased,risk,NaN,NaN,0,NaN,NaN,0,0.817095,0.007960,3,0.812095,0.006780,3
2,bert-base-uncased,topic,0.746511,0.008293,3,0.827287,0.006221,3,NaN,NaN,0,NaN,NaN,0
3,nlpaueb/legal-bert-base-uncased,dual,0.771232,0.002531,3,0.833548,0.002490,3,0.838369,0.002103,3,0.832904,0.001994,3
4,nlpaueb/legal-bert-base-uncased,risk,NaN,NaN,0,NaN,NaN,0,0.823389,0.004344,3,0.818690,0.004177,3
5,nlpaueb/legal-bert-base-uncased,topic,0.777473,0.008943,3,0.833381,0.004093,3,NaN,NaN,0,NaN,NaN,0
6,roberta-base,dual,0.775544,0.006427,3,0.837075,0.002948,3,0.841767,0.005192,3,0.836281,0.005201,3
7,roberta-base,risk,NaN,NaN,0,NaN,NaN,0,0.820242,0.009992,3,0.813223,0.010129,3
8,roberta-base,topic,0.778186,0.006957,3,0.831076,0.003094,3,NaN,NaN,0,NaN,NaN,0
9,xlnet-base-cased,dual,0.761874,0.030934,3,0.834009,0.006944,3,0.835599,0.006967,3,0.829607,0.007767,3


### Bootstrap CIs on the test metrics

Per run, 1,000 clause-level resamples of the test split, computed from the stored logits.
Reported alongside the across-seed sd: the bootstrap CI measures *test-set* sampling
noise, the sd measures *initialisation/ordering* noise. They are different quantities and
the manuscript should not conflate them.

In [12]:
N_RESAMPLES = 1000


def load_run_logits(run_id: str) -> dict:
    payload = np.load(tm.RUNS_DIR / run_id / "test_logits.npz")
    return {
        "topic_logits": payload["topic_logits"],
        "harm_logits": payload["harm_logits"],
        "arrays": {
            "labels": payload["labels"],
            "label_masks": payload["label_masks"],
            "harm_labels": payload["harm_labels"],
            "harm_masks": payload["harm_masks"],
        },
        "row_id": payload["row_id"],
    }


ci_rows = []
for run_id in runs["run_id"]:
    payload = load_run_logits(run_id)
    ci = core.bootstrap_ci(
        payload["topic_logits"], payload["harm_logits"], payload["arrays"], n_resamples=N_RESAMPLES
    )
    ci.insert(0, "run_id", run_id)
    ci_rows.append(ci)

bootstrap_table = pd.concat(ci_rows, ignore_index=True)
bootstrap_table.to_csv(core.EVAL_OUT_DIR / "phase2_bootstrap_ci.csv", index=False)
display(bootstrap_table.head(12))

,run_id,metric,point,mean,ci_low,ci_high,n_rows
0,bert-base-uncased__seed1337__dual,topic_macro_f1,0.768129,0.757783,0.729956,0.782843,2648
1,bert-base-uncased__seed1337__dual,topic_micro_f1,0.833610,0.833595,0.819431,0.846433,2648
2,bert-base-uncased__seed1337__dual,risk_accuracy,0.826284,0.826367,0.811169,0.841399,2648
3,bert-base-uncased__seed1337__dual,risk_macro_f1,0.819565,0.819525,0.803590,0.835370,2648
4,bert-base-uncased__seed1337__risk,topic_macro_f1,0.096569,0.096395,0.092420,0.100558,2648
5,bert-base-uncased__seed1337__risk,topic_micro_f1,0.105170,0.105094,0.101912,0.108382,2648
6,bert-base-uncased__seed1337__risk,risk_accuracy,0.826284,0.826211,0.811178,0.840257,2648
7,bert-base-uncased__seed1337__risk,risk_macro_f1,0.819923,0.819754,0.804343,0.834250,2648
8,bert-base-uncased__seed1337__topic,topic_macro_f1,0.737961,0.735685,0.716233,0.753924,2648
9,bert-base-uncased__seed1337__topic,topic_micro_f1,0.821785,0.821645,0.807027,0.835995,2648


### Paired significance tests

Both tests are **paired on the clause**: every run scored the identical test rows in the
identical order, so a difference is attributable to the varied component and nothing else.

- **Risk head — McNemar.** Item-level correctness per clause (over `harm_mask=1` rows),
  exact binomial on the discordant pairs. This is the right test for two classifiers on
  one sample; an unpaired accuracy comparison would throw away the pairing and lose power.
- **Topic head — paired bootstrap.** Macro-F1 is not decomposable into per-item
  correctness, so McNemar does not apply. Instead each resample draws one set of clause
  indices and scores *both* models on it; the reported interval is over the difference.

Seeds are averaged out by comparing the **best seed** of each arm; change `pick` below to
compare a fixed seed if you would rather not condition on validation performance.

In [13]:
def best_run(encoder: str, heads: str = "dual") -> str:
    subset = runs[(runs["encoder_name"] == encoder) & (runs["heads"] == heads)]
    if subset.empty:
        raise KeyError(f"no runs for {encoder}/{heads}")
    return subset.sort_values("best_val_metric", ascending=False).iloc[0]["run_id"]


def compare(run_a: str, run_b: str) -> dict:
    a, b = load_run_logits(run_a), load_run_logits(run_b)
    assert np.array_equal(a["row_id"], b["row_id"]), "runs were scored on different rows"
    arrays = a["arrays"]

    valid = arrays["harm_masks"].astype(bool)
    correct_a = a["harm_logits"][valid].argmax(1) == arrays["harm_labels"][valid]
    correct_b = b["harm_logits"][valid].argmax(1) == arrays["harm_labels"][valid]
    mcnemar = core.mcnemar(correct_a, correct_b)

    def delta(indices):
        ma = core.topic_metrics(a["topic_logits"][indices], arrays["labels"][indices], arrays["label_masks"][indices])
        mb = core.topic_metrics(b["topic_logits"][indices], arrays["labels"][indices], arrays["label_masks"][indices])
        return ma["topic_macro_f1"] - mb["topic_macro_f1"]

    paired = core.paired_bootstrap_delta(delta, np.arange(len(arrays["labels"])), n_resamples=N_RESAMPLES)
    return {
        "run_a": run_a,
        "run_b": run_b,
        "risk_mcnemar_b": mcnemar["b"],
        "risk_mcnemar_c": mcnemar["c"],
        "risk_mcnemar_p": mcnemar["p_value"],
        "topic_macro_f1_delta": paired["delta"],
        "topic_delta_ci_low": paired["ci_low"],
        "topic_delta_ci_high": paired["ci_high"],
        "topic_delta_p": paired["p_value"],
    }


LEGAL_BERT = tm.ENCODERS[0]
comparisons = []
for other in tm.ENCODERS[1:]:
    try:
        comparisons.append(compare(best_run(LEGAL_BERT), best_run(other)))
    except KeyError as exc:
        print(f"skipped: {exc}")

# Head ablation, original scope: dual vs each single-head variant, legal-bert only.
# Kept verbatim so the Section 4.4.3 numbers stay traceable to the cell that produced them.
for heads in ("topic", "risk"):
    try:
        comparisons.append(compare(best_run(LEGAL_BERT), best_run(LEGAL_BERT, heads)))
    except KeyError as exc:
        print(f"skipped: {exc}")

# Head ablation, extended scope: the same two comparisons for the other three encoders,
# same bootstrap / McNemar apparatus, no new test. Legal-BERT is skipped here because the
# loop above already appended it.
ablation_rows = []
for encoder in tm.ENCODERS:
    for heads in ("topic", "risk"):
        try:
            row = compare(best_run(encoder, "dual"), best_run(encoder, heads))
        except KeyError as exc:
            print(f"skipped: {exc}")
            continue
        ablation_rows.append({"encoder_name": encoder, "heads": heads, **row})
        if encoder != LEGAL_BERT:
            comparisons.append(row)

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(core.EVAL_OUT_DIR / "phase2_head_ablation.csv", index=False)
display(ablation)

significance = pd.DataFrame(comparisons)
significance.to_csv(core.EVAL_OUT_DIR / "phase2_significance.csv", index=False)
display(significance)

,encoder_name,heads,run_a,run_b,risk_mcnemar_b,risk_mcnemar_c,risk_mcnemar_p,topic_macro_f1_delta,topic_delta_ci_low,topic_delta_ci_high,topic_delta_p
0,nlpaueb/legal-bert-base-uncased,topic,legal-bert-base-uncased__seed2024__dual,legal-bert-base-uncased__seed2024__topic,1371,144,2.981594e-251,0.002373,-0.018409,0.024602,0.800
1,nlpaueb/legal-bert-base-uncased,risk,legal-bert-base-uncased__seed2024__dual,legal-bert-base-uncased__seed1337__risk,157,122,4.160536e-02,0.679176,0.641783,0.693528,0.000
2,bert-base-uncased,topic,bert-base-uncased__seed2024__dual,bert-base-uncased__seed2024__topic,1356,137,1.815691e-252,0.025736,-0.001617,0.047774,0.044
3,bert-base-uncased,risk,bert-base-uncased__seed2024__dual,bert-base-uncased__seed42__risk,165,101,1.044672e-04,0.701331,0.660692,0.717730,0.000
4,xlnet-base-cased,topic,xlnet-base-cased__seed2024__dual,xlnet-base-cased__seed2024__topic,1424,153,1.910127e-258,0.001099,-0.017050,0.018725,0.884
5,xlnet-base-cased,risk,xlnet-base-cased__seed2024__dual,xlnet-base-cased__seed42__risk,162,116,6.849966e-03,0.706449,0.665091,0.721040,0.000
6,roberta-base,topic,roberta-base__seed2024__dual,roberta-base__seed1337__topic,1316,123,1.537707e-252,-0.016246,-0.034446,0.004922,0.114
7,roberta-base,risk,roberta-base__seed2024__dual,roberta-base__seed2024__risk,205,103,6.357226e-09,0.678421,0.639655,0.693393,0.000


,run_a,run_b,risk_mcnemar_b,risk_mcnemar_c,risk_mcnemar_p,topic_macro_f1_delta,topic_delta_ci_low,topic_delta_ci_high,topic_delta_p
0,legal-bert-base-uncased__seed2024__dual,bert-base-uncased__seed2024__dual,136,136,1.000000e+00,-0.009945,-0.031126,0.009957,0.329
1,legal-bert-base-uncased__seed2024__dual,xlnet-base-cased__seed2024__dual,123,141,2.954191e-01,-0.018886,-0.036515,-0.000800,0.033
2,legal-bert-base-uncased__seed2024__dual,roberta-base__seed2024__dual,118,147,8.523490e-02,0.002085,-0.018647,0.021480,0.832
3,legal-bert-base-uncased__seed2024__dual,legal-bert-base-uncased__seed2024__topic,1371,144,2.981594e-251,0.002373,-0.018409,0.024602,0.800
4,legal-bert-base-uncased__seed2024__dual,legal-bert-base-uncased__seed1337__risk,157,122,4.160536e-02,0.679176,0.641783,0.693528,0.000
5,bert-base-uncased__seed2024__dual,bert-base-uncased__seed2024__topic,1356,137,1.815691e-252,0.025736,-0.001617,0.047774,0.044
6,bert-base-uncased__seed2024__dual,bert-base-uncased__seed42__risk,165,101,1.044672e-04,0.701331,0.660692,0.717730,0.000
7,xlnet-base-cased__seed2024__dual,xlnet-base-cased__seed2024__topic,1424,153,1.910127e-258,0.001099,-0.017050,0.018725,0.884
8,xlnet-base-cased__seed2024__dual,xlnet-base-cased__seed42__risk,162,116,6.849966e-03,0.706449,0.665091,0.721040,0.000
9,roberta-base__seed2024__dual,roberta-base__seed1337__topic,1316,123,1.537707e-252,-0.016246,-0.034446,0.004922,0.114


## Output table (a) — headline, rows = encoder/config

Cells are `mean +/- sd` over seeds. `n/a` marks a metric a configuration cannot produce:
topic-only leaves the risk head untrained, risk-only leaves the topic head untrained, so
reporting those cells would be reporting random weights.

In [14]:
LABELS = {
    "topic_macro_f1": "Topic macro-F1",
    "topic_micro_f1": "Topic micro-F1",
    "risk_accuracy": "Risk accuracy",
    "risk_macro_f1": "Risk macro-F1",
}
SHORT_NAMES = {
    "nlpaueb/legal-bert-base-uncased": "Legal-BERT",
    "bert-base-uncased": "BERT",
    "xlnet-base-cased": "XLNet",
    "roberta-base": "RoBERTa",
}
HEAD_NAMES = {"dual": "dual", "topic": "topic-only", "risk": "risk-only"}
# 4 encoders x 3 head configs, encoder-major so each encoder's three rows sit together.
CONFIG_NAMES = {
    (encoder, heads): f"{SHORT_NAMES[encoder]} ({HEAD_NAMES[heads]})"
    for encoder in tm.ENCODERS
    for heads in ("dual", "topic", "risk")
}


def mean_sd(values: pd.Series) -> str:
    if values.isna().all():
        return "n/a"
    return f"{values.mean():.3f} ± {values.std(ddof=1):.3f}" if len(values) > 1 else f"{values.mean():.3f}"


headline = pd.DataFrame(
    [
        {
            "Configuration": CONFIG_NAMES.get((encoder, heads), f"{encoder} ({heads})"),
            "Seeds": int(group["seed"].nunique()),
            **{LABELS[m]: mean_sd(group[m]) for m in core.HEADLINE_METRICS},
        }
        for (encoder, heads), group in runs.groupby(["encoder_name", "heads"])
    ]
)
order = [CONFIG_NAMES[k] for k in CONFIG_NAMES if CONFIG_NAMES[k] in set(headline["Configuration"])]
headline = headline.set_index("Configuration").loc[order].reset_index()
display(headline)

core.write_outputs(
    headline,
    "phase2_headline",
    caption=(
        "Test performance by encoder and head configuration, mean $\\pm$ standard deviation "
        "over three seeds (42, 1337, 2024), for the full 4 encoder x 3 head-config design. "
        "All runs use the identical persisted seed-42 "
        "clause split and identical hyperparameters; only the encoder, the seed and the "
        "active heads vary."
    ),
    label="tab:encoder-matrix",
)

,Configuration,Seeds,Topic macro-F1,Topic micro-F1,Risk accuracy,Risk macro-F1
0,Legal-BERT (dual),3,0.771 ± 0.003,0.834 ± 0.002,0.838 ± 0.002,0.833 ± 0.002
1,Legal-BERT (topic-only),3,0.777 ± 0.009,0.833 ± 0.004,n/a,n/a
2,Legal-BERT (risk-only),3,n/a,n/a,0.823 ± 0.004,0.819 ± 0.004
3,BERT (dual),3,0.775 ± 0.006,0.834 ± 0.002,0.830 ± 0.006,0.824 ± 0.006
4,BERT (topic-only),3,0.747 ± 0.008,0.827 ± 0.006,n/a,n/a
5,BERT (risk-only),3,n/a,n/a,0.817 ± 0.008,0.812 ± 0.007
6,XLNet (dual),3,0.762 ± 0.031,0.834 ± 0.007,0.836 ± 0.007,0.830 ± 0.008
7,XLNet (topic-only),3,0.775 ± 0.011,0.834 ± 0.006,n/a,n/a
8,XLNet (risk-only),3,n/a,n/a,0.823 ± 0.005,0.818 ± 0.003
9,RoBERTa (dual),3,0.776 ± 0.006,0.837 ± 0.003,0.842 ± 0.005,0.836 ± 0.005


(WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation/phase2_headline.csv'),
 WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation/phase2_headline.tex'))

## Cross-encoder consistency of the dual-head benefit

The point of running the ablation on all four encoders is not four more rows of numbers —
it is whether the four deltas *agree*. Two views of the same quantity are reported side by
side, and neither is collapsed into a grand mean:

- **Seed-level delta.** Per encoder, dual-head minus single-head on the metric that
  single-head arm still produces (risk macro-F1 for dual-vs-risk-only, topic macro-F1 for
  dual-vs-topic-only), paired by seed, reported as mean +/- sd over the three seed pairs.
  With n = 3 the interval shown is mean +/- t(0.975, 2) * sd / sqrt(3) — wide by
  construction, and that width is the honest statement of what three seeds buy.
- **Best-seed delta.** The McNemar b/c/p (risk) and paired-bootstrap CI (topic) already
  computed in the ablation cell above, on the best-validation seed of each arm. Same
  machinery as every other comparison in this notebook and in the chapter.

The check reported at the end: do all four seed-level intervals overlap a common value,
and does any single encoder's interval sit clear of the other three's range. Overlap
supports "the dual-head benefit is a property of the training objective"; a non-overlapping
encoder rejects it and the claim must be scoped to the backbones where it holds.

In [15]:
# Metric each single-head arm can still be compared on. topic-only leaves the risk head
# untrained and risk-only leaves the topic head untrained, so each contrast has exactly one
# valid metric — the same "n/a" logic as the headline table.
ABLATION_METRIC = {"risk": "risk_macro_f1", "topic": "topic_macro_f1"}
T_CRIT = 4.303  # t(0.975, df=2): three seeds, two-sided 95%


def seed_level_delta(encoder: str, heads: str, metric: str) -> dict:
    """dual minus single-head on `metric`, paired seed by seed."""
    subset = runs[runs["encoder_name"] == encoder]
    dual = subset[subset["heads"] == "dual"].set_index("seed")[metric]
    single = subset[subset["heads"] == heads].set_index("seed")[metric]
    seeds = sorted(set(dual.index) & set(single.index))
    if not seeds:
        raise KeyError(f"no paired seeds for {encoder} dual vs {heads}")
    deltas = np.array([dual[s] - single[s] for s in seeds], dtype=float)
    half_width = T_CRIT * deltas.std(ddof=1) / np.sqrt(len(deltas)) if len(deltas) > 1 else np.nan
    return {
        "encoder": SHORT_NAMES[encoder],
        "contrast": f"dual - {HEAD_NAMES[heads]}",
        "metric": metric,
        "n_seeds": len(deltas),
        "delta_mean": deltas.mean(),
        "delta_sd": deltas.std(ddof=1) if len(deltas) > 1 else np.nan,
        "ci_low": deltas.mean() - half_width,
        "ci_high": deltas.mean() + half_width,
        "all_seeds_positive": bool((deltas > 0).all()),
        "per_seed": np.round(deltas, 4).tolist(),
    }


def overlap_report(table: pd.DataFrame, title: str) -> None:
    """Flag whether the per-encoder intervals share a common value, and name any outlier."""
    print(f"\n{title}")
    for _, row in table.iterrows():
        print(f"  {row['encoder']:<11} {row['delta_mean']:+.4f} +/- {row['delta_sd']:.4f} "
              f"[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]  seeds={row['per_seed']}")

    lower, upper = table["ci_low"].max(), table["ci_high"].min()
    if lower <= upper:
        print(f"  -> all {len(table)} intervals overlap on [{lower:+.4f}, {upper:+.4f}]: "
              "consistent with one common effect.")
    else:
        # No common value. Name the encoders whose interval is disjoint from every other's.
        outliers = [
            row["encoder"] for _, row in table.iterrows()
            if all(row["ci_high"] < o["ci_low"] or row["ci_low"] > o["ci_high"]
                   for _, o in table.iterrows() if o["encoder"] != row["encoder"])
        ]
        print("  -> NO common value: the four intervals do not share a point.")
        print(f"     disjoint from all others: {outliers or 'none individually — pairwise only'}")

    signs = set(np.sign(table["delta_mean"]))
    print(f"     sign agreement: {'all same sign' if len(signs) == 1 else 'SIGNS DISAGREE'}"
          f" ({', '.join(f'{r.encoder} {r.delta_mean:+.4f}' for r in table.itertuples())})")


consistency_rows = []
for heads, metric in ABLATION_METRIC.items():
    for encoder in tm.ENCODERS:
        try:
            consistency_rows.append(seed_level_delta(encoder, heads, metric))
        except KeyError as exc:
            print(f"skipped: {exc}")

consistency = pd.DataFrame(consistency_rows)

# Attach the best-seed significance already computed above, so the seed-level spread and
# the paired test sit in one table instead of two.
if not ablation.empty:
    keyed = ablation.set_index([ablation["encoder_name"].map(SHORT_NAMES),
                                ablation["heads"].map(lambda h: f"dual - {HEAD_NAMES[h]}")])
    for column in ("risk_mcnemar_b", "risk_mcnemar_c", "risk_mcnemar_p",
                   "topic_macro_f1_delta", "topic_delta_ci_low", "topic_delta_ci_high",
                   "topic_delta_p"):
        consistency[column] = [
            keyed[column].get((row.encoder, row.contrast), np.nan) for row in consistency.itertuples()
        ]
    # Blank the columns that do not apply to a contrast: McNemar is a risk-head test, the
    # paired bootstrap a topic-head one.
    risk_rows = consistency["metric"] == "risk_macro_f1"
    consistency.loc[risk_rows, ["topic_macro_f1_delta", "topic_delta_ci_low",
                                "topic_delta_ci_high", "topic_delta_p"]] = np.nan
    consistency.loc[~risk_rows, ["risk_mcnemar_b", "risk_mcnemar_c", "risk_mcnemar_p"]] = np.nan

consistency.to_csv(core.EVAL_OUT_DIR / "phase2_head_ablation_consistency.csv", index=False)
display(consistency)

risk_side = consistency[consistency["metric"] == "risk_macro_f1"].reset_index(drop=True)
topic_side = consistency[consistency["metric"] == "topic_macro_f1"].reset_index(drop=True)
if len(risk_side) > 1:
    overlap_report(risk_side, "Risk macro-F1: dual-head minus risk-only, per encoder")
if len(topic_side) > 1:
    overlap_report(topic_side, "Topic macro-F1: dual-head minus topic-only, per encoder")


,encoder,contrast,metric,n_seeds,delta_mean,delta_sd,ci_low,ci_high,all_seeds_positive,per_seed,risk_mcnemar_b,risk_mcnemar_c,risk_mcnemar_p,topic_macro_f1_delta,topic_delta_ci_low,topic_delta_ci_high,topic_delta_p
0,Legal-BERT,dual - risk-only,risk_macro_f1,3,0.014213,0.002314,0.008465,0.019962,True,"[0.0115, 0.0155, 0.0156]",157.0,122.0,4.160536e-02,NaN,NaN,NaN,NaN
1,BERT,dual - risk-only,risk_macro_f1,3,0.011738,0.011773,-0.017510,0.040985,False,"[0.0124, -0.0004, 0.0232]",165.0,101.0,1.044672e-04,NaN,NaN,NaN,NaN
2,XLNet,dual - risk-only,risk_macro_f1,3,0.012018,0.010523,-0.014125,0.038161,True,"[0.0083, 0.0039, 0.0239]",162.0,116.0,6.849966e-03,NaN,NaN,NaN,NaN
3,RoBERTa,dual - risk-only,risk_macro_f1,3,0.023058,0.015320,-0.015002,0.061118,True,"[0.0108, 0.0182, 0.0402]",205.0,103.0,6.357226e-09,NaN,NaN,NaN,NaN
4,Legal-BERT,dual - topic-only,topic_macro_f1,3,-0.006241,0.007531,-0.024950,0.012468,False,"[-0.0116, -0.0095, 0.0024]",NaN,NaN,NaN,0.002373,-0.018409,0.024602,0.800
5,BERT,dual - topic-only,topic_macro_f1,3,0.028168,0.002247,0.022586,0.033751,True,"[0.0286, 0.0302, 0.0257]",NaN,NaN,NaN,0.025736,-0.001617,0.047774,0.044
6,XLNet,dual - topic-only,topic_macro_f1,3,-0.013143,0.024130,-0.073090,0.046804,False,"[0.0005, -0.041, 0.0011]",NaN,NaN,NaN,0.001099,-0.017050,0.018725,0.884
7,RoBERTa,dual - topic-only,topic_macro_f1,3,-0.002642,0.009383,-0.025953,0.020670,False,"[0.0074, -0.0042, -0.0111]",NaN,NaN,NaN,-0.016246,-0.034446,0.004922,0.114



Risk macro-F1: dual-head minus risk-only, per encoder
  Legal-BERT  +0.0142 +/- 0.0023 [+0.0085, +0.0200]  seeds=[0.0115, 0.0155, 0.0156]
  BERT        +0.0117 +/- 0.0118 [-0.0175, +0.0410]  seeds=[0.0124, -0.0004, 0.0232]
  XLNet       +0.0120 +/- 0.0105 [-0.0141, +0.0382]  seeds=[0.0083, 0.0039, 0.0239]
  RoBERTa     +0.0231 +/- 0.0153 [-0.0150, +0.0611]  seeds=[0.0108, 0.0182, 0.0402]
  -> all 4 intervals overlap on [+0.0085, +0.0200]: consistent with one common effect.
     sign agreement: all same sign (Legal-BERT +0.0142, BERT +0.0117, XLNet +0.0120, RoBERTa +0.0231)

Topic macro-F1: dual-head minus topic-only, per encoder
  Legal-BERT  -0.0062 +/- 0.0075 [-0.0249, +0.0125]  seeds=[-0.0116, -0.0095, 0.0024]
  BERT        +0.0282 +/- 0.0022 [+0.0226, +0.0338]  seeds=[0.0286, 0.0302, 0.0257]
  XLNet       -0.0131 +/- 0.0241 [-0.0731, +0.0468]  seeds=[0.0005, -0.041, 0.0011]
  RoBERTa     -0.0026 +/- 0.0094 [-0.0260, +0.0207]  seeds=[0.0074, -0.0042, -0.0111]
  -> NO common value: 

## Output table (b) — per-topic breakdown for the final model

Rows = 44 topics + macro avg + weighted avg, columns = precision / recall / F1 / support.
Reported twice: for the **best legal-bert seed** (what a deployed single model achieves)
and as the **mean across the three seeds** (what the architecture achieves). Support is
identical across seeds because the split is.

In [16]:
topic_ids, name_by_topic, _ = core.load_taxonomy()

best_legal_bert = best_run(LEGAL_BERT)
best_table = pd.read_csv(tm.RUNS_DIR / best_legal_bert / "per_topic.csv")

seed_tables = [
    pd.read_csv(tm.RUNS_DIR / run_id / "per_topic.csv").set_index("topic_id")
    for run_id in runs[(runs["encoder_name"] == LEGAL_BERT) & (runs["heads"] == "dual")]["run_id"]
]
mean_table = sum(t[["precision", "recall", "f1"]] for t in seed_tables) / len(seed_tables)
mean_table = mean_table.join(seed_tables[0][["support", "observed"]]).reset_index()

per_topic = best_table.merge(mean_table, on="topic_id", suffixes=("_best", "_mean"))
per_topic.insert(1, "topic_name", per_topic["topic_id"].map(lambda t: name_by_topic.get(t, t)))
display(per_topic)

core.write_outputs(
    per_topic[["topic_id", "topic_name", "precision_best", "recall_best", "f1_best",
               "f1_mean", "support_best"]].rename(columns={
        "topic_id": "Topic", "topic_name": "Name", "precision_best": "P", "recall_best": "R",
        "f1_best": "F1", "f1_mean": "F1 (seed mean)", "support_best": "Support"}),
    "phase2_per_topic",
    caption=(
        f"Per-topic test performance of the best Legal-BERT dual-head seed ({best_legal_bert}), "
        "with the mean F1 across the three seeds for comparison. Support counts supervised "
        "positive cells in the test split; topics with no observed test cells are omitted."
    ),
    label="tab:per-topic",
)

,topic_id,topic_name,precision_best,recall_best,f1_best,support_best,observed_best,precision_mean,recall_mean,f1_mean,support_mean,observed_mean
0,choice_of_law,Choice of Law,0.925926,0.840336,0.881057,119,2358,0.912351,0.871148,0.890967,119,2358
1,choice_of_forum,Choice of Forum,0.908257,0.825000,0.864629,120,2360,0.905022,0.872222,0.887903,120,2360
2,mandatory_arbitration,Mandatory Arbitration,0.886792,0.839286,0.862385,56,2357,0.901638,0.875000,0.888003,56,2357
3,class_action_waiver,Class Action Waiver,0.892857,0.877193,0.884956,57,2357,0.914402,0.871345,0.892135,57,2357
4,limitation_of_liability,Limitation of Liability,0.882883,0.837607,0.859649,117,2438,0.885705,0.794872,0.837446,117,2438
5,liability_cap,Liability Cap,0.444444,0.571429,0.500000,7,2336,0.448148,0.428571,0.431624,7,2336
6,warranty_disclaimer,Warranty Disclaimer,0.896970,0.902439,0.899696,164,2336,0.912626,0.902439,0.907183,164,2336
7,indemnification,Indemnification,0.000000,0.000000,0.000000,0,2336,0.000000,0.000000,0.000000,0,2336
8,limitation_period,Limitation Period,1.000000,1.000000,1.000000,1,141,1.000000,1.000000,1.000000,1,141
9,contract_changes,Contract Changes,0.843750,0.900000,0.870968,150,2377,0.842885,0.882222,0.862064,150,2377


(WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation/phase2_per_topic.csv'),
 WindowsPath('C:/Users/Enrique/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation/phase2_per_topic.tex'))